In [0]:
# Step 2: Load ONLY NEW Gold Features (Incremental Processing)
import pandas as pd
from datetime import datetime

# Define checkpoint table to track last processed time
checkpoint_table = 'workspace.server_failure.feature_processing_checkpoint'

# Get last processed timestamp
try:
    last_processed = spark.table(checkpoint_table).select('last_processed_time').first()[0]
    print(f'Last processed time: {last_processed}')
except:
    # First run - no checkpoint exists yet
    last_processed = None
    print('First run - processing all data')

# Read Gold table (full for now, but we'll filter incrementally)
gold_sdf = spark.read.table('workspace.server_failure.server_failure_features_gold')

# For incremental processing:
# If Gold has a timestamp column (e.g., last_event_ts), filter for new data only
if last_processed is not None:
    # Filter for rows with timestamps newer than last checkpoint
    # Assuming 'last_event_ts' exists in Gold - adjust column name if different
    if 'last_event_ts' in gold_sdf.columns:
        gold_sdf = gold_sdf.filter(f"last_event_ts > '{last_processed}'")
        print(f'Reading only data after {last_processed}')
    else:
        print('Warning: No timestamp column found in Gold table - reading all data')

row_count = gold_sdf.count()
print(f'Processing {row_count} rows from Gold table')

# Convert to pandas and fill nulls
features_df = gold_sdf.toPandas()

if len(features_df) > 0:
    for col in features_df.select_dtypes(include='number').columns:
        features_df[col].fillna(features_df[col].mean(), inplace=True)
    for col in features_df.select_dtypes(include='object').columns:
        if len(features_df[col].mode()) > 0:
            features_df[col].fillna(features_df[col].mode()[0], inplace=True)
    print(f'Features loaded and nulls filled: {len(features_df)} rows')
else:
    print('No new data to process')

In [0]:
# Step 3: Create Failure Label
features_df['failure'] = (features_df['crash_events'] > 0).astype(int)

In [0]:
# Step 4: Feature Ranking
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

# Select only numeric columns for X
X = features_df.select_dtypes(include='number').drop(columns=['failure'])
y = features_df['failure']

# Random Forest for feature importance
rf = RandomForestClassifier(n_jobs=-1, random_state=42)
rf.fit(X, y)
rf_importance = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
print('Random Forest Feature Importance:')
print(rf_importance.sort_values(by='importance', ascending=False))

# Mutual Information ranking
mi_scores = mutual_info_classif(X, y, n_jobs=-1)
mi_importance = pd.DataFrame({'feature': X.columns, 'mutual_info': mi_scores})
print('Mutual Information Feature Ranking:')
print(mi_importance.sort_values(by='mutual_info', ascending=False))


In [0]:
# Step 4.5: Drop Unwanted Features
# Drop features with zero mutual information or very low importance

# Identify features with zero mutual information
zero_mi_features = mi_importance[mi_importance['mutual_info'] == 0]['feature'].tolist()

# Identify features with very low RF importance (< 0.03 threshold)
low_importance_features = rf_importance[rf_importance['importance'] < 0.03]['feature'].tolist()

# Combine both lists and get unique features to drop
features_to_drop = list(set(zero_mi_features + low_importance_features))

print(f'Features to drop (low importance/zero MI): {features_to_drop}')

# Drop unwanted features from features_df
features_df_cleaned = features_df.drop(columns=features_to_drop, errors='ignore')

print(f'\nOriginal feature count: {len(features_df.columns)}')
print(f'Cleaned feature count: {len(features_df_cleaned.columns)}')
print(f'Features retained: {list(features_df_cleaned.columns)}')


In [0]:
# Step 5: Store Cleaned Features as Delta Table (Incremental/Streaming-Ready)
from delta.tables import DeltaTable

# Convert cleaned features to Spark DataFrame
features_spark = spark.createDataFrame(features_df_cleaned)

# Define table name
table_name = 'workspace.server_failure.server_failure_features'

# Check if table exists
table_exists = spark.catalog.tableExists(table_name)

if not table_exists:
    # First time: Create table
    features_spark.write \
        .format('delta') \
        .mode('overwrite') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(table_name)
    print(f'Feature table created: {table_name}')
else:
    # Incremental update: Use Delta merge (upsert pattern)
    delta_table = DeltaTable.forName(spark, table_name)
    
    delta_table.alias('target').merge(
        features_spark.alias('source'),
        'target.server_id = source.server_id'
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    print(f'Feature table updated incrementally: {table_name}')

print(f'Total features stored: {len(features_df_cleaned.columns)}')
print(f'Total rows in table: {spark.table(table_name).count()}')
print(f'\nFeature columns: {list(features_df_cleaned.columns)}')